# YZM304 Derin Öğrenme – I. Proje Ödevi
## Meme Kanseri Veri Seti ile Binary Sınıflandırma
**Ankara Üniversitesi | Yapay Zeka ve Veri Mühendisliği | 2025-2026 Bahar**

## 1. Kütüphaneler ve Yapılandırma

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# ─── Global Seed ────────────────────────────────────────────────────────────
SEED       = 42
LR         = 0.01
N_STEPS    = 1000
THRESHOLD  = 0.90   # model seçim eşiği
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Tüm kütüphaneler yüklendi.")
print(f"PyTorch: {torch.__version__}")


## 2. Veri Yükleme ve Keşifsel Veri Analizi (EDA)

In [ ]:
# Veriyi yükle
data = load_breast_cancer()
X_raw = data.data          # (569, 30)
y_raw = data.target        # 0=Malignant, 1=Benign
feature_names = data.feature_names
target_names  = data.target_names

df = pd.DataFrame(X_raw, columns=feature_names)
df['target'] = y_raw

print(f"Veri boyutu     : {X_raw.shape}")
print(f"Sınıf dağılımı  : {dict(zip(*np.unique(y_raw, return_counts=True)))}")
print(f"Sınıf isimleri  : {list(target_names)}")
print("\nİlk 5 satır:")
df.head()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1) Sınıf dağılımı
ax = axes[0, 0]
counts = [np.sum(y_raw==0), np.sum(y_raw==1)]
ax.bar(['Malignant (0)', 'Benign (1)'], counts, color=['#e74c3c','#2ecc71'], edgecolor='black')
ax.set_title('Sınıf Dağılımı', fontsize=13); ax.set_ylabel('Örnek Sayısı')
for i, v in enumerate(counts):
    ax.text(i, v+3, str(v), ha='center', fontweight='bold')

# 2) Özellik ortalamaları (ilk 10)
ax = axes[0, 1]
means_m = df[df.target==0][feature_names[:10]].mean()
means_b = df[df.target==1][feature_names[:10]].mean()
x = np.arange(10)
ax.bar(x-0.2, means_m, 0.4, label='Malignant', color='#e74c3c', alpha=0.8)
ax.bar(x+0.2, means_b, 0.4, label='Benign',    color='#2ecc71', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels([n[:12] for n in feature_names[:10]], rotation=45, ha='right')
ax.set_title('İlk 10 Özellik Ortalamaları'); ax.legend()

# 3) Korelasyon ısı haritası (ilk 10)
ax = axes[1, 0]
corr = df[list(feature_names[:10])+['target']].corr()
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, linewidths=0.5,
            xticklabels=[n[:8] for n in list(feature_names[:10])+['target']],
            yticklabels=[n[:8] for n in list(feature_names[:10])+['target']])
ax.set_title('Korelasyon Isı Haritası (İlk 10 Özellik)')

# 4) Boxplot
ax = axes[1, 1]
df.boxplot(column=list(feature_names[:8]), ax=ax, rot=45, grid=False,
           boxprops=dict(color='#3498db'), medianprops=dict(color='red'))
ax.set_title('Boxplot (İlk 8 Özellik)'); ax.set_xticklabels([n[:10] for n in feature_names[:8]], rotation=45, ha='right')

plt.suptitle('Keşifsel Veri Analizi', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight'); plt.show()
print("eda_plots.png kaydedildi.")


## 3. Ön İşleme: Standardizasyon ve Veri Bölme

In [ ]:
# Standardizasyon
scaler = StandardScaler()
X_all = scaler.fit_transform(X_raw)
y_all = y_raw.reshape(-1, 1).astype(np.float64)

# 70 / 15 / 15  stratified split
X_tmp, X_test,  y_tmp, y_test  = train_test_split(
    X_all, y_all, test_size=0.15, random_state=SEED, stratify=y_all)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, random_state=SEED, stratify=y_tmp)
# 0.1765 * 0.85 ≈ 0.15

print(f"Train : {X_train.shape}  |  Dev : {X_dev.shape}  |  Test : {X_test.shape}")
print(f"Train Benign oranı : {y_train.mean():.3f}")
print(f"Dev   Benign oranı : {y_dev.mean():.3f}")
print(f"Test  Benign oranı : {y_test.mean():.3f}")


## 4. NumPy ile Sinir Ağı (Sıfırdan)

In [ ]:
class NeuralNetwork:
    """
    Tamamen NumPy ile yazılmış çok katmanlı ileri beslemeli sinir ağı.
    Binary Cross-Entropy loss, ReLU gizli aktivasyonlar, Sigmoid çıkış.
    SGD optimizer + L2 Regularizasyon (weight decay).
    """

    def __init__(self, layer_sizes, lr=0.01, n_steps=1000, seed=42, weight_decay=0.0):
        """
        Parametreler
        ------------
        layer_sizes  : list[int]  – ör. [30, 16, 1]
        lr           : float      – öğrenme hızı
        n_steps      : int        – epoch sayısı
        seed         : int        – rastgelelik tohumu
        weight_decay : float      – L2 regülarizasyon katsayısı (lambda)
        """
        self.layer_sizes  = layer_sizes
        self.lr           = lr
        self.n_steps      = n_steps
        self.seed         = seed
        self.weight_decay = weight_decay
        self.params       = {}
        self.history      = {"train_loss": [], "dev_loss": [],
                             "train_acc":  [], "dev_acc":  []}
        self.steps_to_90  = None
        self.__init_weights()

    # ── Private Metodlar ────────────────────────────────────────────────────
    def __init_weights(self):
        """He başlatma ile ağırlıkları başlat."""
        np.random.seed(self.seed)
        for i in range(1, len(self.layer_sizes)):
            fan_in = self.layer_sizes[i-1]
            self.params[f'W{i}'] = np.random.randn(fan_in, self.layer_sizes[i]) * np.sqrt(2.0 / fan_in)
            self.params[f'b{i}'] = np.zeros((1, self.layer_sizes[i]))

    def __relu(self, Z):        return np.maximum(0, Z)
    def __relu_deriv(self, Z):  return (Z > 0).astype(float)
    def __sigmoid(self, Z):     return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))

    def __forward(self, X):
        """İleri geçiş; ara değerleri cache'e atar."""
        cache = {'A0': X}
        n_layers = len(self.layer_sizes) - 1
        for i in range(1, n_layers + 1):
            Z = cache[f'A{i-1}'] @ self.params[f'W{i}'] + self.params[f'b{i}']
            A = self.__sigmoid(Z) if i == n_layers else self.__relu(Z)
            cache[f'Z{i}'] = Z
            cache[f'A{i}'] = A
        return cache

    def __compute_loss(self, y_true, y_pred):
        """Binary Cross-Entropy + L2 regularizasyon penalty."""
        eps = 1e-9
        y_pred = np.clip(y_pred, eps, 1 - eps)
        bce = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        l2_penalty = 0.0
        if self.weight_decay > 0:
            for key, val in self.params.items():
                if key.startswith('W'):
                    l2_penalty += np.sum(val ** 2)
            l2_penalty *= self.weight_decay / (2 * y_true.shape[0])
        return bce + l2_penalty

    def __backward(self, X, y, cache):
        """Geri yayılım; gradyanları hesaplar (L2 ceza dahil)."""
        m = X.shape[0]
        n_layers = len(self.layer_sizes) - 1
        grads = {}
        dA = cache[f'A{n_layers}'] - y
        for i in range(n_layers, 0, -1):
            dW = cache[f'A{i-1}'].T @ dA / m
            if self.weight_decay > 0:
                dW += (self.weight_decay / m) * self.params[f'W{i}']
            db = np.mean(dA, axis=0, keepdims=True)
            grads[f'W{i}'] = dW
            grads[f'b{i}'] = db
            if i > 1:
                dA = (dA @ self.params[f'W{i}'].T) * self.__relu_deriv(cache[f'Z{i-1}'])
        return grads

    def __update_params(self, grads):
        """SGD ağırlık güncelleme."""
        for key in grads:
            self.params[key] -= self.lr * grads[key]

    # ── Public Metodlar ─────────────────────────────────────────────────────
    def fit(self, X_train, y_train, X_dev=None, y_dev=None, verbose=100):
        """Modeli eğit."""
        for step in range(1, self.n_steps + 1):
            cache = self.__forward(X_train)
            grads = self.__backward(X_train, y_train, cache)
            self.__update_params(grads)
            n_layers = len(self.layer_sizes) - 1
            train_loss = self.__compute_loss(y_train, cache[f'A{n_layers}'])
            train_acc  = self.__accuracy(y_train, cache[f'A{n_layers}'])
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            if X_dev is not None:
                dev_cache = self.__forward(X_dev)
                dev_loss  = self.__compute_loss(y_dev, dev_cache[f'A{n_layers}'])
                dev_acc   = self.__accuracy(y_dev, dev_cache[f'A{n_layers}'])
                self.history['dev_loss'].append(dev_loss)
                self.history['dev_acc'].append(dev_acc)
                if self.steps_to_90 is None and dev_acc >= THRESHOLD:
                    self.steps_to_90 = step
            if step % verbose == 0 or step == 1:
                msg = f"  Adım {step:4d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}"
                if X_dev is not None: msg += f" | Dev Loss: {dev_loss:.4f} | Dev Acc: {dev_acc:.4f}"
                print(msg)
        return self

    def predict_proba(self, X):
        """Ham olasılık çıktısı."""
        cache = self.__forward(X)
        return cache[f'A{len(self.layer_sizes)-1}']

    def predict(self, X):
        """Sınıf tahmini (0 veya 1)."""
        return (self.predict_proba(X) >= 0.5).astype(int)

    def evaluate(self, X, y):
        """Accuracy, Precision, Recall, F1 döndürür."""
        y_pred = self.predict(X)
        return {
            "accuracy" : accuracy_score(y, y_pred),
            "precision": precision_score(y, y_pred, zero_division=0),
            "recall"   : recall_score(y, y_pred, zero_division=0),
            "f1"       : f1_score(y, y_pred, zero_division=0),
        }

    @staticmethod
    def __accuracy(y_true, y_pred_prob):
        return np.mean((y_pred_prob >= 0.5).astype(int) == y_true)

print("NeuralNetwork sınıfı tanımlandı.")


### 4.1 Modelleri Eğit

In [ ]:
architectures = {
    "M1_Lab"  : [30, 16, 1],
    "M2_Deeper": [30, 32, 16, 1],
    "M3_Deep"  : [30, 64, 32, 16, 1],
}

trained_models = {}
for name, arch in architectures.items():
    print(f"\n{'='*60}")
    print(f"  Model: {name}  |  Mimari: {arch}  |  L2=0.0 (Regülarizasyon Yok)")
    print('='*60)
    model = NeuralNetwork(layer_sizes=arch, lr=LR, n_steps=N_STEPS, seed=SEED, weight_decay=0.0)
    model.fit(X_train, y_train, X_dev, y_dev, verbose=200)
    trained_models[name] = model
    s90 = model.steps_to_90
    print(f"  ► 90% dev acc eşiğine ulaşılan adım: {s90 if s90 else 'N/A'}")


### 4.2 Overfitting / Underfitting Analizi

In [ ]:
print(f"{'Model':<15} {'Train Acc':>10} {'Dev Acc':>10} {'Test Acc':>10} {'Durum':>15}")
print("-" * 60)
fit_results = {}
for name, model in trained_models.items():
    tr  = model.evaluate(X_train, y_train)
    dv  = model.evaluate(X_dev, y_dev)
    tst = model.evaluate(X_test, y_test)
    gap = tr['accuracy'] - dv['accuracy']
    if   gap > 0.05:  durum = "OverFitting"
    elif tr['accuracy'] < 0.85: durum = "UnderFitting"
    else:              durum = "İyi Fit"
    print(f"{name:<15} {tr['accuracy']:>10.4f} {dv['accuracy']:>10.4f} {tst['accuracy']:>10.4f} {durum:>15}")
    fit_results[name] = {"train": tr, "dev": dv, "test": tst}


### 4.3 Öğrenme Eğrileri

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
colors = {'M1_Lab': '#e74c3c', 'M2_Deeper': '#2ecc71', 'M3_Deep': '#3498db'}
for col, (name, model) in enumerate(trained_models.items()):
    steps = range(1, len(model.history['train_loss']) + 1)
    # Loss
    ax = axes[0, col]
    ax.plot(steps, model.history['train_loss'], color=colors[name], label='Train', lw=1.5)
    ax.plot(steps, model.history['dev_loss'],   color=colors[name], label='Dev',   lw=1.5, ls='--')
    ax.set_title(f'{name} – Loss')
    ax.set_xlabel('Adım'); ax.set_ylabel('BCE Loss'); ax.legend(); ax.grid(alpha=0.3)
    # Accuracy
    ax = axes[1, col]
    ax.plot(steps, model.history['train_acc'], color=colors[name], label='Train', lw=1.5)
    ax.plot(steps, model.history['dev_acc'],   color=colors[name], label='Dev',   lw=1.5, ls='--')
    ax.axhline(THRESHOLD, color='gray', lw=1, ls=':')
    ax.set_title(f'{name} – Accuracy')
    ax.set_xlabel('Adım'); ax.set_ylabel('Accuracy'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Öğrenme Eğrileri (NumPy NN)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('learning_curves.png', dpi=120, bbox_inches='tight'); plt.show()
print("learning_curves.png kaydedildi.")


### 4.4 Model Seçimi

In [ ]:
# Kriter: dev_acc >= THRESHOLD olanlar arasında steps_to_90 en küçük olan
candidates = {n: m for n, m in trained_models.items()
              if m.steps_to_90 is not None and
                 m.evaluate(X_dev, y_dev)['accuracy'] >= THRESHOLD}

if candidates:
    best_name = min(candidates, key=lambda n: candidates[n].steps_to_90)
else:
    # Eşiğe ulaşan yoksa dev_acc en yüksek model
    best_name = max(trained_models, key=lambda n: trained_models[n].evaluate(X_dev, y_dev)['accuracy'])

best_model = trained_models[best_name]
print(f"Seçilen Model : {best_name}")
print(f"Mimari        : {best_model.layer_sizes}")
print(f"Steps-to-90%  : {best_model.steps_to_90}")
print(f"Dev Accuracy  : {best_model.evaluate(X_dev, y_dev)['accuracy']:.4f}")


### 4.5 Regülarizasyon Deneyi: L2 Weight Decay Etkisi

In [ ]:
# En iyi mimari üzerinde farklı L2 katsayıları deneyelim
WD_VALUES = [0.0, 0.001, 0.01, 0.1]
reg_results = {}

print(f"{'L2 (lambda)':<15} {'Train Acc':>10} {'Dev Acc':>10} {'Test Acc':>10} {'Bias/Var Durumu':>18}")
print("-" * 65)

best_arch = [30, 32, 16, 1]  # M2 mimarisi (seçilen model)
for wd in WD_VALUES:
    m = NeuralNetwork(layer_sizes=best_arch, lr=LR, n_steps=N_STEPS,
                      seed=SEED, weight_decay=wd)
    m.fit(X_train, y_train, X_dev, y_dev, verbose=99999)  # sessiz eğitim
    tr  = m.evaluate(X_train, y_train)['accuracy']
    dv  = m.evaluate(X_dev,   y_dev)['accuracy']
    tst = m.evaluate(X_test,  y_test)['accuracy']
    gap = tr - dv
    if   gap > 0.05:  durum = "High Variance (Overfit)"
    elif tr < 0.85:   durum = "High Bias (Underfit)"
    else:             durum = "İyi Fit"
    reg_results[wd] = {'train': tr, 'dev': dv, 'test': tst, 'durum': durum}
    label = f"λ={wd}"
    print(f"{label:<15} {tr:>10.4f} {dv:>10.4f} {tst:>10.4f} {durum:>18}")

print("\nSonuç: L2 regülarizasyonu yüksek varyansı (overfitting'i) azaltır,")
print("       ancak çok büyük λ değerleri yüksek bias'a (underfitting'e) yol açabilir.")


In [ ]:
# Regülarizasyon etkisini görselleştir
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
wds   = list(reg_results.keys())
trs   = [reg_results[w]['train'] for w in wds]
devs  = [reg_results[w]['dev']   for w in wds]
tsts  = [reg_results[w]['test']  for w in wds]
labels = [f'λ={w}' for w in wds]

# Sol: Train vs Dev accuracy
ax = axes[0]
x  = np.arange(len(wds))
w_ = 0.28
ax.bar(x - w_, trs,  w_, label='Train', color='#3498db', alpha=0.85, edgecolor='black')
ax.bar(x,      devs, w_, label='Dev',   color='#e74c3c', alpha=0.85, edgecolor='black')
ax.bar(x + w_, tsts, w_, label='Test',  color='#2ecc71', alpha=0.85, edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0.8, 1.05); ax.set_ylabel('Accuracy'); ax.legend()
ax.set_title('L2 Regülarizasyon: Train / Dev / Test Accuracy', fontsize=12)
ax.grid(axis='y', alpha=0.3)
for xi, (tr_, dv_) in enumerate(zip(trs, devs)):
    ax.text(xi - w_,   tr_  + 0.005, f'{tr_:.3f}',  ha='center', fontsize=7)
    ax.text(xi,        dv_  + 0.005, f'{dv_:.3f}',  ha='center', fontsize=7)

# Sağ: Train-Dev farkı (varyans)
ax2 = axes[1]
gaps = [reg_results[w]['train'] - reg_results[w]['dev'] for w in wds]
colors_gap = ['#e74c3c' if g > 0.05 else '#2ecc71' for g in gaps]
ax2.bar(labels, gaps, color=colors_gap, edgecolor='black', alpha=0.85)
ax2.axhline(0.05, color='red', lw=1.2, ls='--', label='Overfitting eşiği (5%)')
ax2.set_ylabel('Train Acc - Dev Acc (Varyans)')
ax2.set_title('L2 Regülarizasyon: Varyans Analizi', fontsize=12)
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
for xi, g in enumerate(gaps):
    ax2.text(xi, g + 0.001, f'{g:.4f}', ha='center', fontsize=9)

plt.suptitle('Regülarizasyon Etkisi – M2 Mimarisi', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('regularization_analysis.png', dpi=120, bbox_inches='tight'); plt.show()
print("regularization_analysis.png kaydedildi.")


## 5. Scikit-learn MLPClassifier

In [ ]:
# Seçilen modelin mimarisini eşleştir (output nöronu hariç hidden layer'lar)
hidden_layers = tuple(best_model.layer_sizes[1:-1])

# alpha: L2 regularizasyon (weight decay) katsayısı – NumPy modeliyle uyumlu
clf = MLPClassifier(
    hidden_layer_sizes=hidden_layers,
    activation='relu',
    solver='sgd',
    learning_rate_init=LR,
    max_iter=N_STEPS,
    random_state=SEED,
    early_stopping=False,
    validation_fraction=0.0,
    n_iter_no_change=N_STEPS,
    alpha=0.001,   # L2 regülarizasyon – overfitting'i azaltır
    batch_size=32, # Mini-batch SGD
)
clf.fit(X_train, y_train.ravel())

sk_metrics_train = {
    "accuracy" : accuracy_score(y_train, clf.predict(X_train)),
    "precision": precision_score(y_train, clf.predict(X_train), zero_division=0),
    "recall"   : recall_score(y_train, clf.predict(X_train), zero_division=0),
    "f1"       : f1_score(y_train, clf.predict(X_train), zero_division=0),
}
sk_metrics_test = {
    "accuracy" : accuracy_score(y_test, clf.predict(X_test)),
    "precision": precision_score(y_test, clf.predict(X_test), zero_division=0),
    "recall"   : recall_score(y_test, clf.predict(X_test), zero_division=0),
    "f1"       : f1_score(y_test, clf.predict(X_test), zero_division=0),
}
print("Scikit-learn MLP – Eğitim tamamlandı. (L2 alpha=0.001, mini-batch=32)")
print(f"Train Acc : {sk_metrics_train['accuracy']:.4f}")
print(f"Test  Acc : {sk_metrics_test['accuracy']:.4f}")


## 6. PyTorch ile Sinir Ağı

In [ ]:
class PyTorchNN(nn.Module):
    """PyTorch sinir ağı – NumPy modeliyle aynı mimari."""

    def __init__(self, layer_sizes):
        super().__init__()
        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:
                layers.append(nn.ReLU())
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Aynı mimari
pt_model   = PyTorchNN(best_model.layer_sizes)
criterion  = nn.BCELoss()
# weight_decay: L2 regularizasyon; momentum: mini-batch SGD stabilizasyonu
pt_optim   = optim.SGD(pt_model.parameters(), lr=LR, weight_decay=1e-3, momentum=0.9)

# Tensor dönüşümleri
X_tr_t  = torch.tensor(X_train, dtype=torch.float32)
y_tr_t  = torch.tensor(y_train, dtype=torch.float32)
X_tst_t = torch.tensor(X_test,  dtype=torch.float32)
y_tst_t = torch.tensor(y_test,  dtype=torch.float32)
X_dev_t = torch.tensor(X_dev,   dtype=torch.float32)
y_dev_t = torch.tensor(y_dev,   dtype=torch.float32)

# ── Mini-Batch DataLoader ──────────────────────────────────────────────────
BATCH_SIZE = 32
train_dataset = TensorDataset(X_tr_t, y_tr_t)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           generator=torch.Generator().manual_seed(SEED))

print(f"Mini-batch eğitim: batch_size={BATCH_SIZE}, "
      f"toplam {len(train_loader)} batch/epoch")

pt_history = {"train_loss": [], "dev_loss": [], "train_acc": [], "dev_acc": []}
torch.manual_seed(SEED)

for epoch in range(1, N_STEPS + 1):
    pt_model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        pt_optim.zero_grad()
        out  = pt_model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        pt_optim.step()
        epoch_loss += loss.item() * X_batch.size(0)
    epoch_loss /= len(X_tr_t)
    with torch.no_grad():
        pt_model.eval()
        tr_pred  = (pt_model(X_tr_t) >= 0.5).float()
        tr_acc   = (tr_pred == y_tr_t).float().mean().item()
        dev_out  = pt_model(X_dev_t)
        dev_loss = criterion(dev_out, y_dev_t).item()
        dev_pred = (dev_out >= 0.5).float()
        dev_acc  = (dev_pred == y_dev_t).float().mean().item()
        pt_history['train_loss'].append(epoch_loss)
        pt_history['dev_loss'].append(dev_loss)
        pt_history['train_acc'].append(tr_acc)
        pt_history['dev_acc'].append(dev_acc)
    if epoch % 200 == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d} | Train Loss: {epoch_loss:.4f} | Train Acc: {tr_acc:.4f} | Dev Acc: {dev_acc:.4f}")

# PyTorch test metrikleri
with torch.no_grad():
    pt_preds = (pt_model(X_tst_t) >= 0.5).numpy().astype(int)
pt_metrics_test = {
    "accuracy" : accuracy_score(y_test, pt_preds),
    "precision": precision_score(y_test, pt_preds, zero_division=0),
    "recall"   : recall_score(y_test, pt_preds, zero_division=0),
    "f1"       : f1_score(y_test, pt_preds, zero_division=0),
}
print(f"\nPyTorch Test Accuracy: {pt_metrics_test['accuracy']:.4f}")
print(f"(Mini-batch SGD + L2 weight_decay=1e-3 + momentum=0.9 kullanıldı)")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
steps_pt = range(1, N_STEPS + 1)
ax1.plot(steps_pt, pt_history['train_loss'], color='#e67e22', label='Train Loss', lw=1.5)
ax1.plot(steps_pt, pt_history['dev_loss'],   color='#9b59b6', label='Dev Loss',   lw=1.5, ls='--')
ax1.set_title('PyTorch – Loss Eğrisi'); ax1.set_xlabel('Adım'); ax1.set_ylabel('BCE Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(steps_pt, pt_history['train_acc'], color='#e67e22', label='Train Acc', lw=1.5)
ax2.plot(steps_pt, pt_history['dev_acc'],   color='#9b59b6', label='Dev Acc',   lw=1.5, ls='--')
ax2.axhline(THRESHOLD, color='gray', lw=1, ls=':')
ax2.set_title('PyTorch – Accuracy Eğrisi'); ax2.set_xlabel('Adım'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('PyTorch Öğrenme Eğrileri', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('pytorch_curves.png', dpi=120, bbox_inches='tight'); plt.show()
print("pytorch_curves.png kaydedildi.")


## 7. Kapsamlı Değerlendirme

In [ ]:
# Karmaşıklık Matrisleri
numpy_preds = best_model.predict(X_test)
sk_preds    = clf.predict(X_test).reshape(-1, 1)

cms = {
    "NumPy NN" : confusion_matrix(y_test, numpy_preds),
    "Scikit-learn": confusion_matrix(y_test, sk_preds),
    "PyTorch"  : confusion_matrix(y_test, pt_preds),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (title, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['Malignant','Benign'],
                yticklabels=['Malignant','Benign'])
    ax.set_title(title, fontsize=13); ax.set_ylabel('Gerçek'); ax.set_xlabel('Tahmin')
plt.suptitle('Karmaşıklık Matrisleri – Test Seti', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight'); plt.show()
print("confusion_matrices.png kaydedildi.")


In [ ]:
# Metrik Karşılaştırma Tablosu
numpy_test  = best_model.evaluate(X_test, y_test)
results_df  = pd.DataFrame({
    "NumPy NN"   : numpy_test,
    "Scikit-learn": sk_metrics_test,
    "PyTorch"    : pt_metrics_test,
}).T.round(4)
print("\n── Test Seti Metrikleri ──")
print(results_df.to_string())
results_df


In [ ]:
# Bar chart karşılaştırma
metrics_plot = ['accuracy','precision','recall','f1']
models_plot  = ['NumPy NN','Scikit-learn','PyTorch']
vals = {m: [results_df.loc[m, k] for k in metrics_plot] for m in models_plot}

fig, ax = plt.subplots(figsize=(10, 5))
x  = np.arange(len(metrics_plot))
w  = 0.25
cs = ['#3498db','#e74c3c','#2ecc71']
for i, (m, c) in enumerate(zip(models_plot, cs)):
    bars = ax.bar(x + i*w - w, vals[m], w, label=m, color=c, alpha=0.85, edgecolor='black')
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(['Accuracy','Precision','Recall','F1'])
ax.set_ylim(0.5, 1.1); ax.set_ylabel('Skor'); ax.legend()
ax.set_title('Model Metrikleri Karşılaştırması – Test Seti', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight'); plt.show()
print("model_comparison.png kaydedildi.")


In [ ]:
print("\n── Sınıflandırma Raporu (NumPy NN – Seçilen Model) ──")
print(classification_report(y_test, numpy_preds, target_names=['Malignant','Benign']))


## 8. Sonuç Özeti

In [ ]:
print("\n" + "="*70)
print("  YZM304 Derin Öğrenme – I. Proje Ödevi – Sonuç Özeti")
print("="*70)
print(f"  Veri Seti       : Breast Cancer Wisconsin (569 örnek, 30 özellik)")
print(f"  Ön İşleme       : StandardScaler + Stratified Split (70/15/15)")
print(f"  Seçilen Model   : {best_name}  |  Mimari: {best_model.layer_sizes}")
print(f"  Kayıp Fonksiyonu: Binary Cross-Entropy (BCE)")
print(f"  Optimizer       : SGD + L2 Regularizasyon + Mini-Batch (lr={LR})")
print(f"  L2 lambda (NumPy): 0.001 | Scikit-learn alpha=0.001 | PyTorch wd=1e-3")
print(f"  Batch Boyutu    : PyTorch mini-batch={32}, Sklearn mini-batch={32}")
print(f"  Eğitim Adımı    : {N_STEPS}")
print(f"  Steps-to-90%    : {best_model.steps_to_90}")
print("─"*70)
print(f"{'Model':<15} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("─"*70)
for name, row in results_df.iterrows():
    print(f"{name:<15} {row.accuracy:>10.4f} {row.precision:>10.4f} {row.recall:>10.4f} {row.f1:>10.4f}")
print("="*70)
print("  Regülarizasyon deneyi: bkz. Section 4.5 ve regularization_analysis.png")
print("="*70)
